# Import necessary modules

In [43]:
# Standard libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Processing
from sklearn.utils import shuffle
from sklearn.base import clone
from sklearn.model_selection import RepeatedKFold
import random

# Count vectorizer
from sklearn.feature_extraction.text import CountVectorizer

# Metrics
from sklearn.metrics import f1_score, recall_score

# Classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

In [48]:
import pandas as pd
import numpy as np
import os
import random
import pickle
import re
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.svm import SVC
from sklearn.utils import shuffle

# ==============================================================================
# 1. CẤU HÌNH ĐƯỜNG DẪN
# ==============================================================================
RANDOM_STATE = 42
current_dir = os.getcwd()
raw_data_dir = os.path.abspath(os.path.join(current_dir, '..', '..', 'data', 'raw_data'))

print(f"🚀 BẮT ĐẦU QUY TRÌNH LOAD DATA AN TOÀN...")
print(f"📂 Thư mục raw_data: {raw_data_dir}")

# ==============================================================================
# 2. HÀM ĐỌC DỮ LIỆU THÔNG MINH (FIX LỖI KEY ERROR)
# ==============================================================================
def load_robust_data(data_dir):
    all_texts = []
    all_labels = []
    all_ids = []
    
    # Duyệt qua các file split
    for split in ['train', 'dev', 'test']:
        csv_path = os.path.join(data_dir, f'{split}_split_Depression_AVEC2017.csv')
        
        if not os.path.exists(csv_path):
            print(f"⚠️ Bỏ qua: Không tìm thấy file {csv_path}")
            continue
            
        # Đọc file CSV
        try:
            df = pd.read_csv(csv_path)
            # Chuẩn hóa tên cột
            df.columns = [c.strip().lower() for c in df.columns]
            
            # --- TỰ ĐỘNG TÌM CỘT ---
            id_col = next((c for c in df.columns if 'participant' in c), None)
            label_col = next((c for c in df.columns if 'binary' in c), None)
            score_col = next((c for c in df.columns if 'score' in c), None)
            
            # Nếu không thấy cột ID -> Lỗi file
            if id_col is None:
                print(f"❌ LỖI trong file {split}: Không tìm thấy cột ID. Các cột hiện có: {list(df.columns)}")
                continue

            # Xử lý Nhãn (Quan trọng: Nếu thiếu Binary thì tự tính từ Score)
            if label_col is None:
                if score_col is not None:
                    print(f"ℹ️ File {split}: Không có cột Binary, đang tạo nhãn từ cột '{score_col}'...")
                    # PHQ8 >= 10 là trầm cảm
                    label_col = 'calc_binary'
                    df[label_col] = (df[score_col] >= 10).astype(int)
                else:
                    print(f"⚠️ Cảnh báo: File {split} không có cột nhãn (Binary hoặc Score). Bỏ qua file này.")
                    continue
            
            # Bắt đầu đọc Transcript từng người trong file này
            count_ok = 0
            for _, row in df.iterrows():
                pid = int(row[id_col])
                label = int(row[label_col])
                
                # Tìm file transcript
                t_file = os.path.join(data_dir, f'{pid}_P', f'{pid}_TRANSCRIPT.csv')
                
                if os.path.exists(t_file):
                    try:
                        t_df = pd.read_csv(t_file, sep='\t')
                        if 'speaker' in t_df.columns and 'value' in t_df.columns:
                            lines = t_df[t_df['speaker'] == 'Participant']['value'].astype(str)
                            text = " ".join(lines)
                            text = re.sub(r'[^a-zA-Z\s]', '', text.lower()) # Clean text
                            
                            all_texts.append(text)
                            all_labels.append(label)
                            all_ids.append(pid)
                            count_ok += 1
                    except:
                        pass # Bỏ qua lỗi đọc file con để code chạy tiếp
            
            print(f"✅ Đã tải xong {split}: {count_ok} mẫu.")
            
        except Exception as e:
            print(f"❌ Lỗi nghiêm trọng khi đọc file {split}: {e}")

    return all_texts, np.array(all_labels), np.array(all_ids)

# --- THỰC THI ---
texts, y, ids = load_robust_data(raw_data_dir)
print(f"📊 Tổng cộng load được: {len(ids)} mẫu (Train + Dev + Test)")

if len(ids) == 0:
    raise ValueError("❌ Không load được dữ liệu nào! Hãy kiểm tra lại đường dẫn.")

# ==============================================================================
# 3. VECTORIZE & CHIA TRAIN/TEST (CHUẨN XÁC)
# ==============================================================================
print("🔠 Đang tạo Bag of Words...")
CV = CountVectorizer(min_df=3, stop_words='english')
X = CV.fit_transform(texts).toarray()

print("🔪 Đang chia Train/Test...")
# Load danh sách ID Test chuẩn
test_csv = os.path.join(raw_data_dir, 'test_split_Depression_AVEC2017.csv')
if os.path.exists(test_csv):
    test_df_ref = pd.read_csv(test_csv)
    # Lấy cột đầu tiên làm ID (cách an toàn nhất)
    test_ids_ref = test_df_ref.iloc[:, 0].astype(int).values
else:
    raise FileNotFoundError("❌ Thiếu file test_split_Depression_AVEC2017.csv")

X_train, X_test = [], []
y_train, y_test = [], []
id_train, id_test = [], []

for i, pid in enumerate(ids):
    if pid in test_ids_ref:
        X_test.append(X[i])
        y_test.append(y[i])
        id_test.append(pid)
    else:
        X_train.append(X[i])
        y_train.append(y[i])
        id_train.append(pid)

X_train, X_test = np.array(X_train), np.array(X_test)
y_train, y_test = np.array(y_train), np.array(y_test)
id_train, id_test = np.array(id_train), np.array(id_test)

print(f"✅ Train shape: {X_train.shape}")
print(f"✅ Test shape:  {X_test.shape}")

# Chặn lỗi test=0
if len(X_test) == 0:
    # Fallback: Nếu vẫn bằng 0, cắt tạm 20% dữ liệu làm test (chữa cháy)
    print("⚠️ CẢNH BÁO: Không khớp được ID Test nào. Chuyển sang chế độ chia ngẫu nhiên 80-20.")
    from sklearn.model_selection import train_test_split
    X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
        X, y, ids, test_size=0.2, random_state=42, stratify=y
    )
    print(f"✅ New Test shape: {X_test.shape}")

# ==============================================================================
# 4. UNDERSAMPLING, TRAIN & SAVE
# ==============================================================================
print("⚖️ Cân bằng dữ liệu Train...")
idx_0 = [i for i, v in enumerate(y_train) if v == 0]
idx_1 = [i for i, v in enumerate(y_train) if v == 1]
n_min = min(len(idx_0), len(idx_1))

random.seed(RANDOM_STATE)
bal_idx = random.sample(idx_0, n_min) + random.sample(idx_1, n_min)
random.shuffle(bal_idx)

X_train_bal = X_train[bal_idx]
y_train_bal = y_train[bal_idx]

print("🧠 Huấn luyện SVM...")
model = SVC(kernel='linear', C=1, class_weight='balanced', probability=True, random_state=RANDOM_STATE)
model.fit(X_train_bal, y_train_bal)
print(f"✅ Kết quả Test Accuracy: {model.score(X_test, y_test):.4f}")

# Lưu
with open('bow_model.pkl', 'wb') as f:
    pickle.dump(model, f)
np.save('y_text_test_ids.npy', id_test)
np.save('X_test_bow.npy', X_test)

print("="*40)
print("💾 ĐÃ LƯU THÀNH CÔNG (bow_model.pkl, ...)")
print("👉 BẠN CÓ THỂ CHẠY FILE ENSEMBLE ĐƯỢC RỒI!")

🚀 BẮT ĐẦU QUY TRÌNH LOAD DATA AN TOÀN...
📂 Thư mục raw_data: d:\TN-AI\automatic-depression-detector-main\automatic-depression-detector-main\data\raw_data
✅ Đã tải xong train: 107 mẫu.
✅ Đã tải xong dev: 35 mẫu.
⚠️ Cảnh báo: File test không có cột nhãn (Binary hoặc Score). Bỏ qua file này.
📊 Tổng cộng load được: 142 mẫu (Train + Dev + Test)
🔠 Đang tạo Bag of Words...
🔪 Đang chia Train/Test...
✅ Train shape: (142, 2422)
✅ Test shape:  (0,)
⚠️ CẢNH BÁO: Không khớp được ID Test nào. Chuyển sang chế độ chia ngẫu nhiên 80-20.
✅ New Test shape: (29, 2422)
⚖️ Cân bằng dữ liệu Train...
🧠 Huấn luyện SVM...
✅ Kết quả Test Accuracy: 0.7241
💾 ĐÃ LƯU THÀNH CÔNG (bow_model.pkl, ...)
👉 BẠN CÓ THỂ CHẠY FILE ENSEMBLE ĐƯỢC RỒI!


In [49]:
import os
import numpy as np
import pickle
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix

# ==============================================================================
# 1. CẤU HÌNH ĐƯỜNG DẪN (ĐÃ CẬP NHẬT)
# ==============================================================================
current_dir = os.getcwd()

# ĐƯỜNG DẪN AUDIO
AUDIO_MODEL_PATH = os.path.join(current_dir, '..', 'audio', 'audio_model_final.pkl')
AUDIO_FEAT_PATH  = os.path.join(current_dir, '..', 'audio', 'X_audio_mfcc.npy')
AUDIO_IDS_PATH   = os.path.join(current_dir, '..', 'audio', 'y_audio_ids.npy')

# ĐƯỜNG DẪN TEXT (Trỏ vào folder chứa file bạn vừa chạy xong)
# Giả sử bạn đang chạy file ensemble ngay cạnh folder bag_of_words, 
# hoặc sửa đường dẫn '..' tùy vị trí file notebook này.
TEXT_DIR = os.path.join(current_dir, '..', 'bag_of_words') 

TEXT_MODEL_PATH = os.path.join(TEXT_DIR, 'bow_model.pkl')
TEXT_FEAT_PATH  = os.path.join(TEXT_DIR, 'X_test_bow.npy')
TEXT_IDS_PATH   = os.path.join(TEXT_DIR, 'y_text_test_ids.npy')
TEXT_LABEL_PATH = os.path.join(TEXT_DIR, 'y_test_labels.npy') # <-- FILE MỚI LƯU

print("🚀 BẮT ĐẦU ENSEMBLE (INTERSECTION MODE)...")

# ==============================================================================
# 2. HÀM LOAD
# ==============================================================================
def load_model(path):
    with open(path, 'rb') as f: return pickle.load(f)

def load_data_dict(feat_path, ids_path):
    if not os.path.exists(feat_path): return {}
    feats = np.load(feat_path)
    ids = np.load(ids_path)
    return dict(zip(ids.astype(int), feats))

# ==============================================================================
# 3. THỰC THI
# ==============================================================================

# A. Load Models
print("1️⃣ Loading Models...")
if os.path.exists(AUDIO_MODEL_PATH) and os.path.exists(TEXT_MODEL_PATH):
    model_audio = load_model(AUDIO_MODEL_PATH)
    model_text = load_model(TEXT_MODEL_PATH)
    print("   ✅ Models loaded.")
else:
    raise FileNotFoundError("❌ Thiếu file model .pkl (Audio hoặc Text)")

# B. Load Audio Data (Toàn bộ)
print("2️⃣ Loading Audio Data...")
audio_dict = load_data_dict(AUDIO_FEAT_PATH, AUDIO_IDS_PATH)
print(f"   -> Audio database: {len(audio_dict)} patients")

# C. Load Text Test Data (Dữ liệu Test vừa tạo)
print("3️⃣ Loading Text Test Data...")
if os.path.exists(TEXT_FEAT_PATH) and os.path.exists(TEXT_IDS_PATH) and os.path.exists(TEXT_LABEL_PATH):
    text_feats = np.load(TEXT_FEAT_PATH)
    text_ids = np.load(TEXT_IDS_PATH).astype(int)
    text_labels = np.load(TEXT_LABEL_PATH).astype(int)
    
    # Tạo map ID -> (Feature, Label)
    # Vì tập Text Test này có nhãn chuẩn (Ground Truth), ta dùng nó làm mốc
    text_test_map = {}
    for i, pid in enumerate(text_ids):
        text_test_map[pid] = (text_feats[i], text_labels[i])
        
    print(f"   -> Text Test Set: {len(text_test_map)} patients")
else:
    raise FileNotFoundError("❌ Thiếu file dữ liệu Text (.npy). Hãy kiểm tra lại bước trước.")

# ==============================================================================
# 4. CHẠY DỰ ĐOÁN (CHỈ TRÊN NHỮNG ID CHUNG)
# ==============================================================================
print("\n🔄 Đang tìm điểm giao (Intersection) giữa Audio và Text...")

y_true = []
y_pred_ensemble = []
y_pred_audio = []
y_pred_text = []

count_common = 0

for pid, (txt_feat, label) in text_test_map.items():
    # Kiểm tra xem bệnh nhân này có dữ liệu Audio không
    if pid in audio_dict:
        count_common += 1
        
        # 1. Lấy feature
        aud_feat = audio_dict[pid].reshape(1, -1)
        txt_feat = txt_feat.reshape(1, -1)
        
        # 2. Predict Probability
        prob_audio = model_audio.predict_proba(aud_feat)[0][1]
        prob_text = model_text.predict_proba(txt_feat)[0][1]
        
        # 3. Ensemble (Average)
        prob_final = (prob_audio + prob_text) / 2
        
        # 4. Lưu kết quả
        y_true.append(label)
        y_pred_ensemble.append(1 if prob_final > 0.5 else 0)
        y_pred_audio.append(1 if prob_audio > 0.5 else 0)
        y_pred_text.append(1 if prob_text > 0.5 else 0)

print(f"✅ Tìm thấy {count_common} bệnh nhân có đủ dữ liệu cả 2 bên.")

if count_common > 0:
    print("\n" + "="*50)
    print("🏆 KẾT QUẢ SO SÁNH (TRÊN TẬP CHUNG)")
    print("="*50)
    
    f1_a = f1_score(y_true, y_pred_audio)
    f1_t = f1_score(y_true, y_pred_text)
    f1_e = f1_score(y_true, y_pred_ensemble)
    
    acc_e = accuracy_score(y_true, y_pred_ensemble)
    
    print(f"🎵 Audio F1:    {f1_a:.4f}")
    print(f"📝 Text F1:     {f1_t:.4f}")
    print(f"🤝 Ensemble F1: {f1_e:.4f}  <-- KẾT QUẢ CUỐI CÙNG")
    print(f"🎯 Accuracy:    {acc_e:.4f}")
    
    print("\nClassification Report (Ensemble):")
    print(classification_report(y_true, y_pred_ensemble))
    
    if f1_e >= max(f1_a, f1_t):
        print("🎉 Tuyệt vời! Ensemble đã cải thiện hoặc giữ vững hiệu suất.")
    else:
        print("💡 Nhận xét: Ensemble thấp hơn Text đơn lẻ. Điều này thường xảy ra khi Audio quá nhiễu.")
else:
    print("❌ Không có bệnh nhân nào trùng khớp giữa tập Audio và tập Text Test.")

🚀 BẮT ĐẦU ENSEMBLE (INTERSECTION MODE)...
1️⃣ Loading Models...
   ✅ Models loaded.
2️⃣ Loading Audio Data...
   -> Audio database: 188 patients
3️⃣ Loading Text Test Data...
   -> Text Test Set: 29 patients

🔄 Đang tìm điểm giao (Intersection) giữa Audio và Text...
✅ Tìm thấy 29 bệnh nhân có đủ dữ liệu cả 2 bên.

🏆 KẾT QUẢ SO SÁNH (TRÊN TẬP CHUNG)
🎵 Audio F1:    0.5385
📝 Text F1:     0.0000
🤝 Ensemble F1: 0.5385  <-- KẾT QUẢ CUỐI CÙNG
🎯 Accuracy:    0.5862

Classification Report (Ensemble):
              precision    recall  f1-score   support

           0       0.83      0.50      0.62        20
           1       0.41      0.78      0.54         9

    accuracy                           0.59        29
   macro avg       0.62      0.64      0.58        29
weighted avg       0.70      0.59      0.60        29

🎉 Tuyệt vời! Ensemble đã cải thiện hoặc giữ vững hiệu suất.
